# Image Deepfake Detector v9 — Colab Training

**Why:** Previous image model (v8) failed GAS entrance exam at 63.1% (needs ≥80%).
Likely cause: GAS rotated evaluation datasets and our model doesn't generalize.

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells
3. Download `image_detector_v9.zip` when done
4. Push from Ubuntu:
```bash
cd ~/bitmind-subnet && source .venv/bin/activate
python push_standalone.py --image-model ~/Downloads/image_detector_v9.zip --wallet-name miner
```

## Step 1: GPU Check & Dependencies

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('NO GPU! Go to Runtime → Change runtime type → T4 GPU')

!pip install -q timm safetensors

## Step 2: Clone Repo & Get Pretrained Weights

In [ ]:
import os
!git lfs install
!git clone https://github.com/hochul325/deepfake-detector.git /content/repo 2>/dev/null || (cd /content/repo && git pull)

weights_path = '/content/repo/image_detector/model.safetensors'
size_mb = os.path.getsize(weights_path) / 1e6
print(f'Image model weights: {size_mb:.1f} MB')
if size_mb < 10:
    print('LFS pointer! Pulling...')
    !cd /content/repo && git lfs pull
    size_mb = os.path.getsize(weights_path) / 1e6
    print(f'After pull: {size_mb:.1f} MB')

## Step 3: Download Image Datasets via gasbench

In [ ]:
!pip install -q gasbench 2>/dev/null || pip install -q "gasbench @ git+https://github.com/BitMind-AI/gasbench.git@main" 2>/dev/null

GASBENCH_CACHE = '/content/gasbench_cache'
os.makedirs(GASBENCH_CACHE, exist_ok=True)

!gasbench download --modality image --output-dir {GASBENCH_CACHE} --small 2>&1 || echo 'gasbench download command failed'

# Check results
import json
if os.path.exists(GASBENCH_CACHE):
    datasets = [d for d in os.listdir(GASBENCH_CACHE) if os.path.isdir(os.path.join(GASBENCH_CACHE, d))]
    print(f'\nDownloaded {len(datasets)} datasets')
    for d in sorted(datasets):
        info_path = os.path.join(GASBENCH_CACHE, d, 'dataset_info.json')
        if os.path.exists(info_path):
            info = json.load(open(info_path))
            print(f'  {d}: {info.get("modality", "?")} / {info.get("media_type", "?")}')

## Step 4: Configuration

In [ ]:
import os, sys, io, json, random, logging, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from safetensors.torch import save_file, load_file

PRETRAINED_WEIGHTS = '/content/repo/image_detector/model.safetensors'
OUTPUT_DIR = '/content/image_detector_v9'

SEED = 42
EPOCHS = 30
BATCH_SIZE = 32  # T4 16GB
LR = 8e-6
VAL_SPLIT = 0.20

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s',
                    stream=sys.stdout, force=True)
logger = logging.getLogger(__name__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Step 5: Model & Data Pipeline

In [ ]:
class ImageDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2, backbone='vit_base_patch16_clip_224.openai'):
        super().__init__()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.encoder = timm.create_model(backbone, pretrained=True, num_classes=0)
        feature_dim = self.encoder.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim), nn.Dropout(0.3),
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.register_buffer('temperature', torch.ones(1))
        # Freeze early backbone, unfreeze blocks 4-11
        for param in self.encoder.parameters(): param.requires_grad = False
        for name, param in self.encoder.named_parameters():
            if any(f'blocks.{i}.' in name for i in range(4, 12)): param.requires_grad = True
            elif 'norm' in name.lower(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    def forward(self, x):
        x = x / 255.0
        x = (x - self.mean) / self.std
        features = self.encoder(x)
        return self.classifier(features) / self.temperature


class LazyImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try: img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: img = self.transform(img)
        return img, label


class JPEGCompress:
    def __init__(self, qr=(10, 95)): self.qr = qr
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.qr))
        buf.seek(0)
        return Image.open(buf).convert('RGB')

class RandomNoise:
    def __init__(self, sr=(0.005, 0.04)): self.sr = sr
    def __call__(self, img):
        arr = np.array(img, dtype=np.float32) / 255.0
        arr = np.clip(arr + np.random.randn(*arr.shape).astype(np.float32) * random.uniform(*self.sr), 0, 1)
        return Image.fromarray((arr * 255).astype(np.uint8))

class RandomDownscale:
    def __init__(self, sr=(0.25, 0.75)): self.sr = sr
    def __call__(self, img):
        w, h = img.size
        s = random.uniform(*self.sr)
        img = img.resize((max(16, int(w*s)), max(16, int(h*s))), Image.BILINEAR)
        return img.resize((w, h), Image.BILINEAR)


def get_train_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size + 32, size + 32)),
        transforms.RandomCrop((size, size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.08),
        transforms.RandomApply([JPEGCompress()], p=0.45),
        transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.25),
        transforms.RandomApply([RandomNoise()], p=0.2),
        transforms.RandomApply([RandomDownscale()], p=0.2),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
        transforms.Lambda(lambda x: x * 255.0),
    ])

def get_val_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 255.0),
    ])

print('All classes defined.')

## Step 6: Load & Prepare Data

In [ ]:
def load_gasbench_image_datasets():
    cache = Path(GASBENCH_CACHE)
    all_samples = []
    for ds_dir in sorted(cache.iterdir()):
        if not ds_dir.is_dir(): continue
        info_path = ds_dir / 'dataset_info.json'
        week_dirs = [d for d in ds_dir.iterdir() if d.is_dir() and d.name.startswith('20')]
        if week_dirs:
            for week_dir in week_dirs:
                info_path = week_dir / 'dataset_info.json'
                if not info_path.exists(): continue
                with open(info_path) as f: info = json.load(f)
                if info.get('modality') != 'image': continue
                label = 0 if info.get('media_type', '') == 'real' else 1
                ds_name = f'{ds_dir.name}/{week_dir.name}'
                samples_dir = week_dir / 'samples'
                if samples_dir.exists():
                    count = 0
                    for img in sorted(samples_dir.iterdir()):
                        if img.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
                            all_samples.append((str(img), label, ds_name))
                            count += 1
                    print(f'  {ds_name}: {count} {"real" if label==0 else "synth"}')
            continue
        if not info_path.exists(): continue
        with open(info_path) as f: info = json.load(f)
        if info.get('modality') != 'image': continue
        label = 0 if info.get('media_type', '') == 'real' else 1
        ds_name = info.get('name', ds_dir.name)
        samples_dir = ds_dir / 'samples'
        if not samples_dir.exists(): continue
        count = 0
        for img in sorted(samples_dir.iterdir()):
            if img.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
                all_samples.append((str(img), label, ds_name))
                count += 1
        print(f'  {ds_name}: {count} {"real" if label==0 else "synth"}')
    return all_samples


print('Loading image datasets...')
all_samples = load_gasbench_image_datasets()
n_real = sum(1 for _, l, _ in all_samples if l == 0)
n_synth = sum(1 for _, l, _ in all_samples if l == 1)
print(f'\nTotal: {len(all_samples)} ({n_real} real, {n_synth} synth)')

if len(all_samples) == 0:
    print('\nERROR: No images found! Check gasbench download.')
else:
    # Oversample small datasets to at least 200
    ds_counts = Counter(ds for _, _, ds in all_samples)
    balanced = []
    for ds_name, count in ds_counts.items():
        items = [(p, l, d) for p, l, d in all_samples if d == ds_name]
        if count < 200:
            items = items * max(1, 200 // count)
            print(f'  Oversampled {ds_name}: {count} -> {len(items)}')
        balanced.extend(items)
    all_samples = balanced
    n_real = sum(1 for _, l, _ in all_samples if l == 0)
    n_synth = sum(1 for _, l, _ in all_samples if l == 1)
    print(f'After balancing: {len(all_samples)} ({n_real} real, {n_synth} synth)')

    random.shuffle(all_samples)
    val_size = int(len(all_samples) * VAL_SPLIT)
    val_with_names = all_samples[:val_size]
    train_with_names = all_samples[val_size:]
    train_samples = [(p, l) for p, l, _ in train_with_names]
    val_samples = [(p, l) for p, l, _ in val_with_names]
    print(f'Train: {len(train_samples)}, Val: {len(val_samples)}')

In [ ]:
# Create dataloaders
train_ds = LazyImageDataset(train_samples, get_train_transform())
val_ds = LazyImageDataset(val_samples, get_val_transform())

train_labels = [l for _, l in train_samples]
cc = [train_labels.count(0), train_labels.count(1)]
print(f'Class counts: real={cc[0]}, synth={cc[1]}')

if min(cc) > 0:
    weights = [1.0 / cc[l] for l in train_labels]
    sampler = WeightedRandomSampler(weights, len(train_labels), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True, drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}')

## Step 7: Create Model & Load Weights

In [ ]:
model = ImageDeepfakeDetector().to(device)

if os.path.exists(PRETRAINED_WEIGHTS):
    print('Loading pretrained weights...')
    state = load_file(PRETRAINED_WEIGHTS)
    state = {k: v for k, v in state.items() if k != 'temperature'}
    model.load_state_dict(state, strict=False)
    print('Loaded!')
else:
    print('No pretrained weights — training from CLIP ViT scratch')

## Step 8: Train

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, epoch, mixup_alpha=0.3):
    model.train()
    weight = torch.tensor([1.0, 1.15]).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.08, weight=weight)
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        if mixup_alpha > 0 and random.random() < 0.4:
            lam = max(l := np.random.beta(mixup_alpha, mixup_alpha), 1 - l)
            idx = torch.randperm(images.size(0), device=device)
            logits = model(lam * images + (1 - lam) * images[idx])
            loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[idx])
        else:
            logits = model(images)
            loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        if (batch_idx + 1) % 50 == 0:
            print(f'Ep{epoch} [{batch_idx+1}/{len(loader)}] Loss:{total_loss/(batch_idx+1):.4f} Acc:{100*correct/total:.1f}%')
    return total_loss / max(len(loader), 1), correct / max(total, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        logits = model(images.to(device))
        probs = F.softmax(logits, dim=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    labels = np.array(all_labels); preds = np.array(all_preds); probs = np.array(all_probs)
    acc = (preds == labels).mean()
    tp=((preds==1)&(labels==1)).sum(); tn=((preds==0)&(labels==0)).sum()
    fp=((preds==1)&(labels==0)).sum(); fn=((preds==0)&(labels==1)).sum()
    d = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
    mcc = float(tp*tn - fp*fn) / d if d > 0 else 0
    brier = float(np.mean((probs[:, 1] - labels) ** 2))
    mn = max(0, ((mcc+1)/2))**1.2; bn = max(0, (0.25-brier)/0.25)**1.8
    sn34 = float(np.sqrt(max(1e-12, mn*bn)))
    print(f'  Acc:{acc:.4f} MCC:{mcc:.4f} Brier:{brier:.6f} sn34:{sn34:.4f} (TP={tp} TN={tn} FP={fp} FN={fn})')
    return {'accuracy': acc, 'mcc': mcc, 'brier': brier, 'sn34_score': sn34}

print('Training functions ready.')

In [ ]:
# === TRAINING LOOP ===
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' in n]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR},
    {'params': classifier_params, 'lr': LR * 10},
], weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = len(train_loader) * 2

def lr_lambda(step):
    if step < warmup_steps: return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_sn34, best_epoch = 0, 0
patience, patience_counter = 8, 0

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*50}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*50}')
    loss, acc = train_epoch(model, train_loader, optimizer, scheduler, device, epoch)
    print(f'Train Loss:{loss:.4f} Acc:{acc:.4f}')
    metrics = evaluate(model, val_loader, device)
    if metrics['sn34_score'] > best_sn34:
        best_sn34 = metrics['sn34_score']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), '/content/best_image_v9.pt')
        print(f'  *** New best sn34: {best_sn34:.4f} ***')
    else:
        patience_counter += 1
        if patience_counter >= patience and epoch > 5:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest epoch: {best_epoch} (sn34: {best_sn34:.4f})')

## Step 9: Calibrate & Package

In [ ]:
model.load_state_dict(torch.load('/content/best_image_v9.pt', weights_only=True))

# Temperature calibration
model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    model.temperature.fill_(1.0)
    for images, labels in val_loader:
        all_logits.append(model(images.to(device)).cpu())
        all_labels.append(labels)
logits = torch.cat(all_logits); labels = torch.cat(all_labels)

best_brier, best_temp = float('inf'), 1.0
for temp in np.arange(0.1, 10.0, 0.02):
    probs = F.softmax(logits / temp, dim=1)
    brier = ((probs[:, 1] - labels.float()) ** 2).mean().item()
    if brier < best_brier: best_brier, best_temp = brier, temp
for temp in np.arange(max(0.05, best_temp-0.15), best_temp+0.15, 0.001):
    probs = F.softmax(logits / temp, dim=1)
    brier = ((probs[:, 1] - labels.float()) ** 2).mean().item()
    if brier < best_brier: best_brier, best_temp = brier, temp

print(f'Calibrated: temp={best_temp:.3f} brier={best_brier:.6f}')
model.temperature.fill_(best_temp)

print('\nFinal evaluation:')
final = evaluate(model, val_loader, device)

In [ ]:
# Package as GAS-ready zip
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
save_file(model.state_dict(), str(out / 'model.safetensors'))

(out / 'model_config.yaml').write_text("""name: "clip-vit-image-deepfake-detector-gas"
version: "4.0.0"
modality: "image"

preprocessing:
  resize: [224, 224]

model:
  num_classes: 2
  weights_file: "model.safetensors"
""")

(out / 'model.py').write_text('''import torch
import torch.nn as nn
import timm
from safetensors.torch import load_file


class ImageDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.encoder = timm.create_model("vit_base_patch16_clip_224.openai", pretrained=False, num_classes=0)
        feature_dim = self.encoder.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim), nn.Dropout(0.3),
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.register_buffer("temperature", torch.ones(1))

    def forward(self, x):
        x = x.float() / 255.0
        x = (x - self.mean) / self.std
        features = self.encoder(x)
        return self.classifier(features) / self.temperature


def load_model(weights_path, num_classes=2):
    model = ImageDeepfakeDetector(num_classes=num_classes)
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.train(False)
    return model
''')

zip_path = '/content/image_detector_v9.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(out / 'model.safetensors', 'model.safetensors')
    zf.write(out / 'model_config.yaml', 'model_config.yaml')
    zf.write(out / 'model.py', 'model.py')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'\nDone! {zip_path} ({size_mb:.1f} MB)')
print(f'Final sn34: {final["sn34_score"]:.4f}')
print(f'\nDownload zip, then on Ubuntu:')
print(f'  cd ~/bitmind-subnet && source .venv/bin/activate')
print(f'  python push_standalone.py --image-model image_detector_v9.zip --wallet-name miner')

In [ ]:
from google.colab import files
files.download('/content/image_detector_v9.zip')